# **Kaggle – DataTops®**
Tu TA ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Aspectos importantes
- Última submission:
    - Mañana: 17 de febrero a las 5pm
    - Tarde: 19 de febrero a las 5pm
- **Enlace de la competición**: https://www.kaggle.com/t/c5cc87b50c4b4770bdc8f5acbe15577d
- **Requisito**: Estar registrado en [Kaggle](https://www.kaggle.com/)

## Métrica:
El error cuadrático medio (RMSE, por sus siglas en inglés) es una medida de la desviación estándar de los residuos (errores de predicción). Los residuos representan la diferencia entre los valores observados y los valores predichos por el modelo. El RMSE indica qué tan dispersos están estos errores: cuanto menor es el RMSE, más cercanas están las predicciones a los valores reales. En otras palabras, el RMSE mide qué tan bien se ajusta la línea de regresión a los datos.


$$ RMSE = \sqrt{\frac{1}{n}\Sigma_{i=1}^{n}{\Big(\frac{d_i -f_i}{\sigma_i}\Big)^2}}$$


## 1. Librerías

In [1]:
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
import urllib.request

## 2. Datos

In [2]:
# Para que funcione necesitas bajarte los archivos de datos de Kaggle
df = pd.read_csv("./data/train.csv")

### 2.1 Exploración de los datos

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 912 entries, 0 to 911
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   laptop_ID         912 non-null    int64  
 1   Company           912 non-null    object 
 2   Product           912 non-null    object 
 3   TypeName          912 non-null    object 
 4   Inches            912 non-null    float64
 5   ScreenResolution  912 non-null    object 
 6   Cpu               912 non-null    object 
 7   Ram               912 non-null    object 
 8   Memory            912 non-null    object 
 9   Gpu               912 non-null    object 
 10  OpSys             912 non-null    object 
 11  Weight            912 non-null    object 
 12  Price_in_euros    912 non-null    float64
dtypes: float64(2), int64(1), object(10)
memory usage: 92.8+ KB


In [4]:
df.head()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
0,755,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.86kg,539.00
1,618,Dell,Inspiron 7559,Gaming,15.6,Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,16GB,1TB HDD,Nvidia GeForce GTX 960<U+039C>,Windows 10,2.59kg,879.01
2,909,HP,ProBook 450,Notebook,15.6,Full HD 1920x1080,Intel Core i7 7500U 2.7GHz,8GB,1TB HDD,Nvidia GeForce 930MX,Windows 10,2.04kg,900.00
3,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
4,286,Dell,Inspiron 3567,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,AMD Radeon R5 M430,Linux,2.25kg,428.00


In [5]:
df.tail()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
907,28,Dell,Inspiron 5570,Notebook,15.6,Full HD 1920x1080,Intel Core i5 8250U 1.6GHz,8GB,256GB SSD,AMD Radeon 530,Windows 10,2.2kg,800.00
908,1160,HP,Spectre Pro,2 in 1 Convertible,13.3,Full HD / Touchscreen 1920x1080,Intel Core i5 6300U 2.4GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.48kg,1629.00
909,78,Lenovo,IdeaPad 320-15IKBN,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,2TB HDD,Intel HD Graphics 620,No OS,2.2kg,519.00
910,23,HP,255 G6,Notebook,15.6,1366x768,AMD E-Series E2-9000e 1.5GHz,4GB,500GB HDD,AMD Radeon R2,No OS,1.86kg,258.00
911,229,Dell,Alienware 17,Gaming,17.3,IPS Panel Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,256GB SSD + 1TB HDD,Nvidia GeForce GTX 1060,Windows 10,4.42kg,2456.34


In [6]:
df.describe()

,laptop_ID,Inches,Price_in_euros
count,912.000000,912.000000,912.000000
mean,650.312500,14.981579,1111.724090
std,382.727748,1.436719,687.959172
min,2.000000,10.100000,174.000000
25%,324.750000,14.000000,589.000000
50%,636.500000,15.600000,978.000000
75%,982.250000,15.600000,1483.942500
max,1320.000000,18.400000,6099.000000


In [7]:
df.isnull().sum()

laptop_ID           0
Company             0
Product             0
TypeName            0
Inches              0
ScreenResolution    0
Cpu                 0
Ram                 0
Memory              0
Gpu                 0
OpSys               0
Weight              0
Price_in_euros      0
dtype: int64

### Análisis de variables categóricas y cardinalidad

Antes de realizar transformaciones, se analiza la cardinalidad de las variables categóricas para identificar posibles problemas que puedan afectar al rendimiento del modelo.

In [8]:
categorical_cols = df.select_dtypes(include=["object"]).columns
df[categorical_cols].nunique().sort_values(ascending=False)

Product             480
Weight              165
Cpu                 107
Gpu                  93
Memory               37
ScreenResolution     36
Company              19
Ram                   9
OpSys                 9
TypeName              6
dtype: int64

Tras analizar la cardinalidad, `Product` tiene 480 valores únicos sobre 912 observaciones (más del 50%). Su codificación generaría demasiadas dummies con pocas observaciones, lo que induciría sobreajuste. Por eso se elimina.


El identificador `laptop_ID` se descarta por no tener capacidad predictiva.

In [9]:
df.drop(columns=["laptop_ID", "Product"], inplace=True)


In [10]:
#Transformacion de variables

# RAM
df["Ram"] = df["Ram"].str.replace("GB", "", regex=False).astype(int)

# Peso
df["Weight"] = df["Weight"].str.replace("kg", "", regex=False).astype(float)

# Resolución de pantalla
resolution = df["ScreenResolution"].str.extract(r"(\d+)x(\d+)")
df["ScreenResolution"] = resolution[0].astype(int) * resolution[1].astype(int)


In [11]:
# Simplificación de CPU y GPU

df["Cpu_brand"] = df["Cpu"].str.split().str[0]
df.drop(columns=["Cpu"], inplace=True)

df["Gpu_brand"] = df["Gpu"].str.split().str[0]
df.drop(columns=["Gpu"], inplace=True)


In [12]:
# SSD vs HDD — suele influir bastante en el precio
df["SSD"] = df["Memory"].str.contains("SSD", case=False, na=False).astype(int)
df["HDD"] = df["Memory"].str.contains("HDD", case=False, na=False).astype(int)

# GB totales de almacenamiento
mem = df["Memory"].str.extract(r"(\d+(?:\.\d+)?)(GB|TB)")
mem_val = pd.to_numeric(mem[0], errors="coerce")
df["Storage_GB"] = np.where(mem[1] == "TB", mem_val * 1024, mem_val)

df.drop(columns=["Memory"], inplace=True)


### 2.3 Definir X e y

In [13]:
X = df.drop(['Price_in_euros'], axis=1)
y = df['Price_in_euros'].copy()
X.shape

(912, 12)

In [14]:
y.shape

(912,)

In [15]:
# Identificación de features numéricas y categóricas

features_num = X.select_dtypes(include=["int64", "float64"]).columns
features_cat = X.select_dtypes(include=["object"]).columns

features_num, features_cat

(Index(['Inches', 'ScreenResolution', 'Ram', 'Weight', 'SSD', 'HDD',
        'Storage_GB'],
       dtype='object'),
 Index(['Company', 'TypeName', 'OpSys', 'Cpu_brand', 'Gpu_brand'], dtype='object'))

In [16]:
X[features_num].describe()

,Inches,ScreenResolution,Ram,Weight,SSD,HDD,Storage_GB
count,912.000000,9.120000e+02,912.000000,912.000000,912.000000,912.000000,912.000000
mean,14.981579,2.139313e+06,8.263158,2.026937,0.643640,0.434211,436.192982
std,1.436719,1.347720e+06,5.044788,0.665466,0.479186,0.495925,355.394517
min,10.100000,1.049088e+06,2.000000,0.690000,0.000000,0.000000,8.000000
25%,14.000000,1.234272e+06,4.000000,1.490000,0.000000,0.000000,256.000000
50%,15.600000,2.073600e+06,8.000000,2.040000,1.000000,0.000000,256.000000
75%,15.600000,2.073600e+06,8.000000,2.300000,1.000000,1.000000,512.000000
max,18.400000,8.294400e+06,64.000000,4.700000,1.000000,1.000000,2048.000000


In [17]:
#Codificación de variables categóricas
X = pd.get_dummies(X, drop_first=True)

### 2.4 Dividir X_train, X_test, y_train, y_test

In [23]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [24]:
X_train

,Inches,ScreenResolution,Ram,Weight,SSD,HDD,Storage_GB,Company_Apple,Company_Asus,Company_Chuwi,...,OpSys_Linux,OpSys_Mac OS X,OpSys_No OS,OpSys_Windows 10,OpSys_Windows 10 S,OpSys_Windows 7,OpSys_macOS,Cpu_brand_Intel,Gpu_brand_Intel,Gpu_brand_Nvidia
25,17.3,2073600,8,3.00,0,1,1024.0,False,False,False,...,False,False,False,False,False,True,False,True,False,False
84,15.6,2073600,16,2.56,1,0,512.0,False,False,False,...,False,False,False,True,False,False,False,True,False,True
10,13.3,4096000,8,1.37,1,0,512.0,True,False,False,...,False,False,False,False,False,False,True,True,True,False
342,14.0,2073600,4,1.54,0,1,500.0,False,False,False,...,False,False,False,False,False,True,False,True,True,False
890,17.3,2073600,16,2.80,1,1,256.0,False,False,False,...,False,False,False,True,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,14.0,1049088,8,1.94,0,1,2048.0,False,False,False,...,False,False,False,True,False,False,False,True,True,False
270,15.6,2073600,6,2.20,1,0,256.0,False,False,False,...,False,False,False,True,False,False,False,False,False,False
860,12.5,2073600,16,1.18,1,0,256.0,False,False,False,...,False,False,False,True,False,False,False,True,True,False
435,15.6,1049088,4,2.20,0,1,1024.0,False,False,False,...,False,False,False,True,False,False,False,True,True,False


In [25]:
y_train

25     2899.00
84     1249.26
10     1958.90
342    1030.99
890    1396.00
        ...   
106     389.00
270     549.00
860    1859.00
435     306.00
102    1943.00
Name: Price_in_euros, Length: 729, dtype: float64

## 3. Procesado de datos

Nuestro target es la columna `Price_in_euros`

-----------------------------------------------------------------------------------------------------------------

## 4. Modelado

### 4.1 Baseline de modelos


In [26]:
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "SVR": SVR()
}


### 4.2 Sacar métricas, valorar los modelos

Recuerda que en la competición se va a evaluar con la métrica de ``RMSE``.

In [27]:
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    rmse = root_mean_squared_error(y_val, preds)
    results[name] = rmse
    print(f"{name} RMSE: {rmse}")

results


LinearRegression RMSE: 388.2910196651952
RandomForest RMSE: 358.0199533703487
SVR RMSE: 745.6781107817451


{'LinearRegression': np.float64(388.2910196651952),
 'RandomForest': np.float64(358.0199533703487),
 'SVR': np.float64(745.6781107817451)}

### 4.3 Optimización (up to you 🫰🏻)

In [28]:
# RandomForest con hiperparámetros ajustados
rf_optimized = RandomForestRegressor(
    n_estimators=400,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_optimized.fit(X_train, y_train)
y_pred_rf = rf_optimized.predict(X_val)
rmse_rf = root_mean_squared_error(y_val, y_pred_rf)
print(f"RandomForest optimizado RMSE: {rmse_rf}")

RandomForest optimizado RMSE: 349.3507948017053


In [29]:
# GradientBoosting — entrena de forma secuencial y suele capturar mejor los patrones complejos
gb_model = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    random_state=42
)

gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_val)
rmse_gb = root_mean_squared_error(y_val, y_pred_gb)
print(f"GradientBoosting RMSE: {rmse_gb}")

GradientBoosting RMSE: 329.4573458617217


In [30]:
# Elijo el modelo con menor RMSE
print(f"RF optimizado:    {rmse_rf:.2f} €")
print(f"GradientBoosting: {rmse_gb:.2f} €")


RF optimizado:    349.35 €
GradientBoosting: 329.46 €


In [31]:
# Me quedo con GradientBoosting y entreno con todos los datos
final_model = gb_model
final_model.fit(X, y)

GradientBoostingRegressor(learning_rate=0.05, max_depth=4, n_estimators=500,
                          random_state=42, subsample=0.8)

-----------------------------------------------------------------

## Una vez listo el modelo, toca predecir ``test.csv``

**RECUERDA: APLICAR LAS TRANSFORMACIONES QUE HAYAS REALIZADO EN `train.csv` a `test.csv`.**


Véase:
- Estandarización/Normalización
- Eliminación de Outliers
- Eliminación de columnas
- Creación de columnas nuevas
- Gestión de valores nulos
- Y un largo etcétera de técnicas que como Data Scientist hayas considerado las mejores para tu dataset.

## 1. Carga los datos de `test.csv` para predecir.


In [32]:
X_pred = pd.read_csv("./data/test.csv")
X_pred.head()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
0,209,Lenovo,Legion Y520-15IKBN,Gaming,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD,Nvidia GeForce GTX 1060,No OS,2.4kg
1,1281,Acer,Aspire ES1-531,Notebook,15.6,1366x768,Intel Celeron Dual Core N3060 1.6GHz,4GB,500GB HDD,Intel HD Graphics 400,Linux,2.4kg
2,1168,Lenovo,V110-15ISK (i3-6006U/4GB/1TB/No,Notebook,15.6,1366x768,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,Intel HD Graphics 520,No OS,1.9kg
3,1231,Dell,Inspiron 7579,2 in 1 Convertible,15.6,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,2.191kg
4,1020,HP,ProBook 640,Notebook,14.0,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,4GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.95kg


In [33]:
X_pred.tail()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
386,820,MSI,GE72MVR 7RG,Gaming,17.3,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD + 1TB HDD,Nvidia GeForce GTX 1070,Windows 10,2.9kg
387,948,Toshiba,Tecra Z40-C-12X,Notebook,14.0,IPS Panel Full HD 1920x1080,Intel Core i5 6200U 2.3GHz,4GB,128GB SSD,Intel HD Graphics 520,Windows 10,1.47kg
388,483,Dell,Precision M5520,Workstation,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,8GB,256GB SSD,Nvidia Quadro M1200,Windows 10,1.78kg
389,1017,HP,Probook 440,Notebook,14.0,1366x768,Intel Core i5 7200U 2.5GHz,4GB,500GB HDD,Intel HD Graphics 620,Windows 10,1.64kg
390,421,Asus,ZenBook Flip,2 in 1 Convertible,13.3,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.27kg


In [34]:
X_pred.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 391 entries, 0 to 390
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   laptop_ID         391 non-null    int64  
 1   Company           391 non-null    object 
 2   Product           391 non-null    object 
 3   TypeName          391 non-null    object 
 4   Inches            391 non-null    float64
 5   ScreenResolution  391 non-null    object 
 6   Cpu               391 non-null    object 
 7   Ram               391 non-null    object 
 8   Memory            391 non-null    object 
 9   Gpu               391 non-null    object 
 10  OpSys             391 non-null    object 
 11  Weight            391 non-null    object 
dtypes: float64(1), int64(1), object(10)
memory usage: 36.8+ KB


 ## 2. Replicar el procesado para ``test.csv``

In [35]:
X_pred

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
0,209,Lenovo,Legion Y520-15IKBN,Gaming,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD,Nvidia GeForce GTX 1060,No OS,2.4kg
1,1281,Acer,Aspire ES1-531,Notebook,15.6,1366x768,Intel Celeron Dual Core N3060 1.6GHz,4GB,500GB HDD,Intel HD Graphics 400,Linux,2.4kg
2,1168,Lenovo,V110-15ISK (i3-6006U/4GB/1TB/No,Notebook,15.6,1366x768,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,Intel HD Graphics 520,No OS,1.9kg
3,1231,Dell,Inspiron 7579,2 in 1 Convertible,15.6,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,2.191kg
4,1020,HP,ProBook 640,Notebook,14.0,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,4GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.95kg
...,...,...,...,...,...,...,...,...,...,...,...,...
386,820,MSI,GE72MVR 7RG,Gaming,17.3,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD + 1TB HDD,Nvidia GeForce GTX 1070,Windows 10,2.9kg
387,948,Toshiba,Tecra Z40-C-12X,Notebook,14.0,IPS Panel Full HD 1920x1080,Intel Core i5 6200U 2.3GHz,4GB,128GB SSD,Intel HD Graphics 520,Windows 10,1.47kg
388,483,Dell,Precision M5520,Workstation,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,8GB,256GB SSD,Nvidia Quadro M1200,Windows 10,1.78kg
389,1017,HP,Probook 440,Notebook,14.0,1366x768,Intel Core i5 7200U 2.5GHz,4GB,500GB HDD,Intel HD Graphics 620,Windows 10,1.64kg


In [36]:
# Preprocesamiento del conjunto de test — mismas transformaciones que en train

test_df = X_pred.copy()

test_df.drop(columns=["laptop_ID", "Product"], inplace=True)

test_df["Ram"] = test_df["Ram"].str.replace("GB", "", regex=False).astype(int)
test_df["Weight"] = test_df["Weight"].str.replace("kg", "", regex=False).astype(float)

resolution = test_df["ScreenResolution"].str.extract(r"(\d+)x(\d+)")
test_df["ScreenResolution"] = resolution[0].astype(int) * resolution[1].astype(int)

test_df["Cpu_brand"] = test_df["Cpu"].str.split().str[0]
test_df.drop(columns=["Cpu"], inplace=True)

test_df["Gpu_brand"] = test_df["Gpu"].str.split().str[0]
test_df.drop(columns=["Gpu"], inplace=True)

# Feature engineering igual que en train
test_df["SSD"] = test_df["Memory"].str.contains("SSD", case=False, na=False).astype(int)
test_df["HDD"] = test_df["Memory"].str.contains("HDD", case=False, na=False).astype(int)

mem_test = test_df["Memory"].str.extract(r"(\d+(?:\.\d+)?)(GB|TB)")
mem_val_test = pd.to_numeric(mem_test[0], errors="coerce")
test_df["Storage_GB"] = np.where(mem_test[1] == "TB", mem_val_test * 1024, mem_val_test)

test_df.drop(columns=["Memory"], inplace=True)

In [37]:
#Codificación del test y alineación con train

test_df = pd.get_dummies(test_df, drop_first=True)
test_df = test_df.reindex(columns=X.columns, fill_value=0)


In [38]:
predictions_submit = final_model.predict(test_df)
predictions_submit


array([1363.12964581,  376.74616364,  346.86573375,  993.68994806,
       1057.14420054,  582.16838187,  634.09457023, 1063.20076344,
       1268.95077389,  295.32377328, 2193.97576751, 1442.72947085,
        452.74151821, 1956.63166119,  722.64274937,  530.12967425,
       3255.71674183, 1340.95993342, 1905.50352033,  638.55806777,
       1468.37174747,  345.38756562,  787.79920367, 1054.35436721,
        482.34919289,  723.61753453,  622.75214935,  871.18367561,
       2769.57997979, 1119.14021856, 1957.09303235,  425.74790369,
        873.19133793, 3034.74014436, 2416.04654253, 1981.57933758,
        502.12716202, 1399.82684775,  953.54755515, 1760.41245049,
        550.01340734,  946.43475209,  538.32679027, 1278.65427422,
       1049.44897568, 1127.01775766, 1147.72087699,  556.70274543,
        735.86110612,  538.9242927 , 1611.03419134,  773.96964046,
       1103.38617953,  350.81585568, 1812.51511719, 1624.79433441,
        630.36476363,  894.72058155, 1297.84100239,  703.23843

**¡OJO! ¿Por qué me da error?**

IMPORTANTE:

- SI EL ARRAY CON EL QUE HICISTEIS `.fit()` ERA DE 4 COLUMNAS, PARA `.predict()` DEBEN SER LAS MISMAS
- SI AL ARRAY CON EL QUE HICISTEIS `.fit()` LO NORMALIZASTEIS, PARA `.predict()` DEBÉIS NORMALIZARLO
- TODO IGUAL SALVO **BORRAR FILAS**, EL NÚMERO DE ROWS SE DEBE MANTENER EN ESTE SET, PUES LA PREDICCIÓN DEBE TENER **391 FILAS**, SI O SI

**Entonces, si al cargar los datos de ``train.csv`` usaste `index_col=0`, ¿tendré que hacer lo también para el `test.csv`?**

In [39]:
# ¿Qué opináis?
# ¿Sí, no?

## 3. **¿Qué es lo que subirás a Kaggle?**

**Para subir a Kaggle la predicción esta tendrá que tener una forma específica.**

En este caso, la **MISMA** forma que `sample_submission.csv`.

In [46]:
sample = pd.read_csv("data/sample_submission.csv")
submission_df = pd.read_csv("data/sample_submission.csv")

In [47]:
sample.head()

,laptop_ID,Price_in_euros
0,209,1949.1
1,1281,805.0
2,1168,1101.0
3,1231,1293.8
4,1020,1832.6


In [48]:
sample.shape

(391, 2)

## 4. Mete tus predicciones en un dataframe llamado ``submission``.

In [49]:
#¿Cómo creamos la submission?
submission = pd.DataFrame()

In [50]:
# Crear el DataFrame de submission a partir del ejemplo
submission = submission_df.copy()

# Sustituir la columna de precios por nuestras predicciones
submission.iloc[:, 1] = predictions_submit.astype(float)

In [51]:
submission.head()

,laptop_ID,Price_in_euros
0,209,1363.129646
1,1281,376.746164
2,1168,346.865734
3,1231,993.689948
4,1020,1057.144201


In [52]:
submission.shape

(391, 2)

## 5. Pásale el CHEQUEADOR para comprobar que efectivamente está listo para subir a Kaggle.

In [53]:
def chequeador(df_to_submit):
    """
    Esta función se asegura de que tu submission tenga la forma requerida por Kaggle.

    Si es así, se guardará el dataframe en un `csv` y estará listo para subir a Kaggle.

    Si no, LEE EL MENSAJE Y HAZLE CASO.

    Si aún no:
    - apaga tu ordenador,
    - date una vuelta,
    - enciendelo otra vez,
    - abre este notebook y
    - leelo todo de nuevo.
    Todos nos merecemos una segunda oportunidad. También tú.
    """
    if df_to_submit.shape == sample.shape:
        if df_to_submit.columns.all() == sample.columns.all():
            if df_to_submit.laptop_ID.all() == sample.laptop_ID.all():
                print("You're ready to submit!")
                df_to_submit.to_csv("submission.csv", index = False) #muy importante el index = False
                urllib.request.urlretrieve("https://www.mihaileric.com/static/evaluation-meme-e0a350f278a36346e6d46b139b1d0da0-ed51e.jpg", "gfg.png")
                img = Image.open("gfg.png")
                img.show()
            else:
                print("Check the ids and try again")
        else:
            print("Check the names of the columns and try again")
    else:
        print("Check the number of rows and/or columns and try again")
        print("\nMensaje secreto del TA: No me puedo creer que después de todo este notebook hayas hecho algún cambio en las filas de `test.csv`. Lloro.")

In [54]:
chequeador(submission)

You're ready to submit!
